# Week 8a.1: Multi-Agent Orchestration: Agent Swarms

The Swarm Pattern is a decentralized multi-agent orchestration approach in which specialized agents coordinate by directly handing off control to one another, without a central supervisor. Each agent autonomously decides which agent should act next based on the current task and shared context.

In this notebook, we'll split the Husky Tech support agent into two agents. 
- The **orders** agent will handle order status, shipping, warranty, and support hours. 
- The **returns** agent will handle returns and refunds. Each agent can hand the customer to the other with a tool call, and there is no supervisor. This arrangement is called a **swarm**.

Two distinguishing properties:
- The agent that is talking decides who talks next, by calling a handoff tool.
- The graph remembers which agent was talking, and the customer's next message goes to that agent.


<img src="figures/support_swarm_subgraphs.png" width="900">

## 0. Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. Two agents

The four tools are the same ones from W03b, which are available in `support_tools.py`. Each agent will get the tools for its own job:

| Agent | Tools | Handles |
|---|---|---|
| `orders` | `look_up_order`, `search_faq` | order status, shipping, warranty, support hours |
| `returns` | `check_return_eligibility`, `escalate_to_human` | returns and refunds |

A customer can ask about both in one message. Each prompt tells its agent to do its own part first and then hand off, and to answer every part when it replies, because only the last agent's reply reaches the customer.

In [ ]:
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

from support_tools import look_up_order, search_faq, check_return_eligibility, escalate_to_human

ORDERS_PROMPT = """You are the orders agent for Husky Tech, an online electronics store. You work with a returns agent.

You handle order status, shipping, warranty, and support hours. The returns agent handles returns and refunds.

Rules:
- If the customer's latest message is only about a return or a refund, call transfer_to_returns right away. Do not answer it yourself.
- If it asks about a return and also about something you handle, look up your part first, then call transfer_to_returns.
- When you reply to the customer, answer every part of their latest message, using what is already in the conversation.
- Never invent order details. Use look_up_order and search_faq.
- Be concise and polite."""

RETURNS_PROMPT = """You are the returns agent for Husky Tech, an online electronics store. You work with an orders agent.

You handle returns and refunds. The orders agent handles order status, shipping, warranty, and support hours.

Rules:
- If the customer's latest message is only about something other than a return or a refund, call transfer_to_orders right away. Do not answer it yourself.
- If it asks about a return and also about something the orders agent handles, handle the return first, then call transfer_to_orders.
- When a customer wants to return an order, call check_return_eligibility with the order id from the conversation before you reply.
- If the order is not eligible, or the customer asks for a person, use escalate_to_human.
- When you reply to the customer, answer every part of their latest message, using what is already in the conversation.
- Be concise and polite."""

Each prompt tells the agent what it handles, what the other agent handles, and when to hand over.

The instruction "Do not answer it yourself" matters: 
- without it, an agent asked about something outside its job tends to answer anyway

## 2. Handoff tools

**A handoff is a tool call.** 

The tool itself does nothing except return a message. Inside the agent, the agent's graph sees that a handoff tool ran and stops the agent's loop. Outside, the swarm graph reads the tool's name and starts the other agent.

The returned string is the tool's result, and it is written for the receiving agent, not the customer. The customer never sees it. It goes into the message list, where it is the last thing the receiving agent reads before it replies, so it speaks to that agent directly: "You are now the returns agent. Answer the customer's latest message." 

Without the second sentence, a small model often greets the customer as if the conversation had just started, and when the message only reports the handoff ("Handed off to the returns agent."), a small model tends to repeat it to the customer instead of answering.

In [ ]:
@tool
def transfer_to_returns() -> str:
    """Hand the customer to the returns agent, for returns and refunds."""
    return "You are now the returns agent. Answer the customer's latest message."


@tool
def transfer_to_orders() -> str:
    """Hand the customer to the orders agent, for order status, shipping, warranty, and support hours."""
    return "You are now the orders agent. Answer the customer's latest message."


HANDOFFS = {"transfer_to_returns": "returns", "transfer_to_orders": "orders"}

In [ ]:
orders_tools = [look_up_order, search_faq, transfer_to_returns]
returns_tools = [check_return_eligibility, escalate_to_human, transfer_to_orders]

## 3. Each agent is a graph

Each agent is the W04b support agent with two changes:

- Its `assistant` node also writes the agent's name to `active_agent`, so the state records who spoke last.
- After the tools run, the agent checks whether the tool that just ran was a handoff. If it was, the agent's graph ends instead of going back to `assistant`, so the agent never answers a customer it has just handed over.


Both agents use the state class `SwarmState`, which is `MessagesState` with `active_agent` added. The swarm graph uses the same class, so when an agent finishes, its messages and its `active_agent` go straight into the swarm's state.

In [ ]:
class SwarmState(MessagesState):
    active_agent: str


def build_agent(name, prompt, tools):
    # model_with_tools = model.bind_tools(tools, parallel_tool_calls=False)
    model_with_tools = model.bind_tools(tools)
    sys_msg = SystemMessage(content=prompt)

    def assistant(state: SwarmState) -> dict:
        result = [model_with_tools.invoke([sys_msg] + state["messages"])]
        return {"messages": result, "active_agent": name}

    # TODO add after_tools to route to END if a handoff runs. Otherwise back to assistant

    builder = StateGraph(SwarmState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant") #TODO replace with conditional routing for handoffs
    return builder.compile()


orders_agent = build_agent("orders", ORDERS_PROMPT, orders_tools)
returns_agent = build_agent("returns", RETURNS_PROMPT, returns_tools)

In [ ]:
display(Image(orders_agent.get_graph().draw_mermaid_png()))

In [ ]:
def show(messages):
    for m in messages:
        calls = [c["name"] for c in getattr(m, "tool_calls", None) or []]
        print(f"  {m.type:5} {(m.text or '')[:90]!r} {calls or ''}")



In [ ]:
message = [HumanMessage(content="My headphones from HT-1001 stopped working. Can I return them?")]
result = orders_agent.invoke({"messages": message})
show(result["messages"])

## 4. The swarm graph

The swarm graph has one node per agent.

There are two routing decisions:

- **From `START`**: go to `active_agent`, or to `orders` on the first turn. This is how the customer's next message reaches the agent that was talking.
- **After an agent finishes**: if its last message is the result of a handoff tool, go to the agent that tool names. Otherwise the agent has answered the customer, and the turn ends.

In [ ]:
def start_route(state: SwarmState) -> str:
    # TODO: update active_agent (default: orders)


def after_agent(state: SwarmState) -> str:
    last = state["messages"][-1]
    # TODO if the last message was a handoff return the corresponding node name
    return END

In [ ]:
swarm = StateGraph(SwarmState)
swarm.add_node("orders", orders_agent)
swarm.add_node("returns", returns_agent)

swarm.add_conditional_edges(START, start_route, ["orders", "returns"])
swarm.add_conditional_edges("orders", after_agent, ["returns", END])
swarm.add_conditional_edges("returns", after_agent, ["orders", END])

swarm_agent = swarm.compile(checkpointer=MemorySaver())

display(Image(swarm_agent.get_graph().draw_mermaid_png()))

Each agent is a single node in the swarm graph. The edges between `orders` and `returns` are the handoffs, and every edge is dashed because `start_route` and `after_agent` decide them. 

The checkpointer goes on the swarm graph only; the subgraphs are part of its run.

## 5. Let's give it a run

With `subgraphs=True`, `stream` reports the steps inside each agent too, tagged with the agent they ran in, so a step prints as `orders.tools`. 

When a handoff tool is called with a `reason`, `chat` also prints a `handoff:` line with the reason.

In [ ]:
def chat(graph, text, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    path, handoffs = [], []
    
    for namespace, update in graph.stream({"messages": [HumanMessage(content=text)]}, config,
                                          stream_mode="updates", subgraphs=True):
        if namespace:                                   # a step inside one of the agents
            agent = namespace[0].split(":")[0]
            for node, values in update.items():
                path.append(f"{agent}.{node}")
                for m in (values or {}).get("messages", []):
                    for call in getattr(m, "tool_calls", None) or []:
                        if call["name"].startswith("transfer_to") and "reason" in call["args"]:
                            handoffs.append(f"{agent} -> {call['name']}: {call['args']['reason']}")
    
    state = graph.get_state(config).values

    print(f"customer: {text}")
    print(f"path:     {' -> '.join(path)}")
    for line in handoffs:
        print(f"handoff:  {line}")
    print(f"[{state.get('active_agent', 'orders')}] {state['messages'][-1].text}\n")

In [ ]:
chat(swarm_agent, "Where is my order HT-1002?", "customer-1")

In [ ]:
chat(swarm_agent, "My headphones from HT-1001 stopped working. Can I return them?", "customer-1")

In [ ]:
chat(swarm_agent, "Thanks. Also, how long does shipping usually take?", "customer-1")

In [ ]:
state = swarm_agent.get_state({"configurable": {"thread_id": "customer-1"}}).values
print("active agent:", state["active_agent"])
for m in state["messages"]:
    calls = [c["name"] for c in getattr(m, "tool_calls", None) or []]
    print(f"  {m.type:5} {(m.text or '')[:70]!r} {calls or ''}")

The returns agent saw the orders agent's lookup of HT-1002, and the orders agent saw the escalation. In a swarm every agent reads the whole conversation. That is what lets the receiving agent pick up in the middle of a conversation, and it is also why each prompt has to say clearly which questions belong to which agent.

## 6. Handoffs with `Command`

A handoff took two steps in two graphs. 
- Inside the agent, `after_tools` saw that a handoff ran and ended the agent's loop. 
- Outside, `after_agent` read the tool's name and started the other agent.

LangGraph lets the handoff tool do both at once, by returning a `Command` instead of a string. 

A `Command` carries an update to the state and the node to run next:

```python
return Command(update={"active_agent": "returns"}, goto="returns", graph=Command.PARENT)
```

- The update is written to the state, and the graph runs the `goto` node next.
    - Whatever returns the `Command` chooses the next step itself, so no edge is needed to get there.
- `graph=Command.PARENT` says which graph `goto` refers to: the graph that contains this agent.
    - Our handoff tool runs inside the orders agent's graph, but `returns` is a node of the swarm graph, which is one level up.

A tool that returns a `Command` also has to write two things:

- **Its tool result.** A tool that returns a string gets its `ToolMessage` from `ToolNode`. A tool that returns a `Command` writes its own, with the `tool_call_id` of the call it answers
    - The model APIs reject a history where a tool call has no result.
- **The agent's messages.** When an agent finishes normally, its messages go into the swarm's state. A `Command` to the parent leaves the agent's graph before that happens, so the update carries the agent's messages along with the new `ToolMessage`. 
    - Messages the swarm already has are matched by their id, so none is added twice.

`ToolRuntime` gives the tool both: `runtime.tool_call_id` and `runtime.state`.
- The model never sees the `runtime` argument; `ToolNode` fills it in when the tool runs.


In [ ]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime
from langgraph.types import Command


@tool
def transfer_to_returns(reason: str, runtime: ToolRuntime) -> Command:
    """Hand the customer to the returns agent, for returns and refunds. Give a one-sentence reason."""
    note = ToolMessage(content=f"You are now the returns agent. The orders agent handed over because: {reason} "
                               "Answer the customer's latest message.",
                       tool_call_id=runtime.tool_call_id)
    return Command(#TODO)


@tool
def transfer_to_orders(reason: str, runtime: ToolRuntime) -> Command:
    """Hand the customer to the orders agent, for order status, shipping, warranty, and support hours. Give a one-sentence reason."""
    note = ToolMessage(content=f"You are now the orders agent. The returns agent handed over because: {reason} "
                               "Answer the customer's latest message.",
                       tool_call_id=runtime.tool_call_id)
    return Command(#TODO)

The tools also take a `reason`, as `escalate_to_human` does. When a model calls a tool, its message usually has no text, only the call, so nothing records why it handed the customer over. A `reason` argument makes the model write one sentence of explanation as part of the call. The tool puts it in the `ToolMessage`, so the receiving agent also learns why it has the customer.

### Updated Graph

The swarm graph will no longer have `after_agent`. The agents have no edges out of them: a handoff goes where its `Command` says, and an agent that finishes without a handoff has nothing after it, so the turn ends. `destinations` lists where each agent's handoff tools can go. LangGraph uses it only to draw those edges.

A handoff that leaves with `Command.PARENT` has to be the only tool call in its step. A model can ask for several tools in one message: on "Where is my order HT-1002, and can I return the headphones from HT-1001?" the orders agent may call `look_up_order` and `transfer_to_returns` together. Both run, but the handoff leaves the agent's graph before `look_up_order`'s result is saved, and the next model call fails because a tool call in the history has no result.

`parallel_tool_calls=False` tells the model to make at most one tool call per message. 
- OpenAI models support it. `ChatOllama` and `ChatGoogleGenerativeAI` do not accept it, so remove it if you use one of those, and expect an occasional error on a question that needs both agents.

In [ ]:
def build_agent_cmd(name, prompt, tools):
    #model_with_tools = model.bind_tools(tools, parallel_tool_calls=False)   # OpenAI only; remove for Ollama or Gemini
    model_with_tools = model.bind_tools(tools)
    sys_msg = SystemMessage(content=prompt)

    def assistant(state: SwarmState) -> dict:
        return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])], "active_agent": name}

    builder = StateGraph(SwarmState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile()


orders_cmd = build_agent_cmd("orders", ORDERS_PROMPT, [look_up_order, search_faq, transfer_to_returns])
returns_cmd = build_agent_cmd("returns", RETURNS_PROMPT, [check_return_eligibility, escalate_to_human, transfer_to_orders])


In [ ]:
swarm = StateGraph(SwarmState)
swarm.add_node("orders", orders_cmd, destinations=("returns",))
swarm.add_node("returns", returns_cmd, destinations=("orders",))
#TODO add conditional edge for routing to agents

swarm_command = swarm.compile(checkpointer=MemorySaver())

In [ ]:
display(Image(swarm_command.get_graph().draw_mermaid_png()))

In [ ]:
for text in ["Where is my order HT-1002?",
             "My headphones from HT-1001 stopped working. Can I return them?",
             "Thanks. Also, how long does shipping usually take?"]:
    chat(swarm_command, text, "customer-3")

In [ ]:
chat(swarm_command, "Where is my keyboard order HT-1002, and can I return the headphones from HT-1001?", "customer-4")

## 7. LangGraph Libraries for Multi-Agent Systems

LangGraph provides built-in functions for common multi-agent patterns.

`langgraph-swarm` builds the same swarm from a list of agents. Each agent has to be a compiled graph with a name, built with LangChain's `create_agent`, which is `build_agent_cmd`
- https://reference.langchain.com/python/langgraph-swarm

`create_handoff_tool` writes the handoff tools: each returns `Command(goto=..., graph=Command.PARENT)` with the agent's messages and a `ToolMessage`. 
- https://reference.langchain.com/python/langgraph-supervisor/handoff/create_handoff_tool

`create_swarm` adds each agent as a node with its destinations, and routes from `START` by `active_agent`. It does not turn off parallel tool calls, so the failure described in §6 can happen with it too. `create_supervisor` turns them off by default.

Also check out `create_supervisor`:
- https://reference.langchain.com/python/langgraph-supervisor

In [ ]:
from langchain.agents import create_agent
from langgraph_swarm import create_swarm, create_handoff_tool

to_returns = create_handoff_tool(agent_name="returns",
                                 description="Hand the customer to the returns agent, for returns and refunds.")
to_orders = create_handoff_tool(agent_name="orders",
                                description="Hand the customer to the orders agent, for order status, shipping, "
                                            "warranty, and support hours.")

orders_agent = create_agent(model, tools=[look_up_order, search_faq, to_returns],
                            name="orders", system_prompt=ORDERS_PROMPT)
returns_agent = create_agent(model, tools=[check_return_eligibility, escalate_to_human, to_orders],
                             name="returns", system_prompt=RETURNS_PROMPT)

desk_lib = create_swarm([orders_agent, returns_agent], default_active_agent="orders").compile(checkpointer=MemorySaver())

for text in ["Where is my order HT-1002?",
             "My headphones from HT-1001 stopped working. Can I return them?",
             "Thanks. Also, how long does shipping usually take?"]:
    chat(desk_lib, text, "customer-2")



| | Our implementation | `langgraph-swarm` |
|---|---|---|
| an agent | `build_agent_cmd`: the W04b loop, compiled | `create_agent(model, tools, name=..., system_prompt=...)` |
| a handoff | a tool that returns `Command(goto=..., graph=Command.PARENT)` | `create_handoff_tool(agent_name=...)` |
| the swarm | one node per agent with `destinations`, and `start_route` from `START` | `create_swarm(agents, default_active_agent=...)` |
| who was talking | `active_agent`, set by `assistant` and by the handoff tool | `active_agent`, set by the handoff tool |

Every row is something you wrote in §6. When the library does something you did not expect, you know where to look in its code.

## 8. Things to try out

1. Add a third agent, `warranty`, for items that are broken but past the return window. Give it `search_faq` and `escalate_to_human`, give `returns` a handoff tool to it, and give it a handoff tool back to `orders`. 

2. Remove "Do not answer it yourself." from the returns prompt, rebuild the Section 6 swarm, and run the three turns on a new thread. Does turn 3 still reach `orders`?

3. In Section 6, change the update in `transfer_to_returns` to `{"messages": [note], "active_agent": "returns"}`, so it carries only the `ToolMessage`. Rebuild, and run turn 2 on a new thread. What goes wrong, and which message is missing from the history?

In [ ]:
# your experiments here